# Iris Classifier using Vertex AI


## Overview

In this tutorial, you build a scikit-learn model and deploy it on infer in local environment using Google Cloud Storage for logging and tracking model and data


### Dataset

This tutorial uses R.A. Fisher's Iris dataset, a small and popular dataset for machine learning experiments. Each instance has four numerical features, which are different measurements of a flower, and a target label that
categorizes the flower into: **Iris setosa**, **Iris versicolour** and **Iris virginica**.

This tutorial uses [a version of the Iris dataset available in the
scikit-learn library](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_iris.html#sklearn.datasets.load_iris).

### Costs

This tutorial uses billable components of Google Cloud:

* Vertex AI
* Cloud Storage

Learn about [Vertex AI
pricing](https://cloud.google.com/vertex-ai/pricing), [Cloud Storage
pricing](https://cloud.google.com/storage/pricing), 

## Get started

### Install Vertex AI SDK for Python and other required packages



In [10]:
# Vertex SDK for Python
! pip3 install --upgrade --quiet  google-cloud-aiplatform

### Set Google Cloud project information
Learn more about [setting up a project and a development environment](https://cloud.google.com/vertex-ai/docs/start/cloud-environment).

In [11]:
PROJECT_ID = "mystical-atlas-473806-i7"  # @param {type:"string"}
LOCATION = "asia-south1"  # @param {type:"string"}

### Create a Cloud Storage bucket

Create a storage bucket to store intermediate artifacts such as datasets.

In [16]:
from datetime import datetime
BUCKET_NAME = "mlops-course-mystical-atlas-473806-i7-unique"
BUCKET_URI = f"gs://{BUCKET_NAME}"  # @param {type:"string"}
BASE_MODEL_ARTIFACT_DIR="iris_classifier/model"

**If your bucket doesn't already exist**: Run the following cell to create your Cloud Storage bucket.

In [16]:
! gsutil mb -l {LOCATION} -p {PROJECT_ID} {BUCKET_URI}

Creating gs://mlops-course-mystical-atlas-473806-i7-unique/...


### Initialize Vertex AI SDK for Python

To get started using Vertex AI, you must have an existing Google Cloud project and [enable the Vertex AI API](https://console.cloud.google.com/flows/enableapi?apiid=aiplatform.googleapis.com).

In [17]:
from google.cloud import aiplatform

aiplatform.init(project=PROJECT_ID, location=LOCATION, staging_bucket=BUCKET_URI)

### Import the required libraries

In [18]:
import os
import sys

## Simple Decision Tree model
Build a Decision Tree model on iris data

In [9]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from pandas.plotting import parallel_coordinates
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn import metrics


In [2]:
import pickle
import joblib

def pipeline(data_path, model_name):
    data = pd.read_csv(data_path)
    
    train, test = train_test_split(data, test_size = 0.4, stratify = data['species'], random_state = 42)
    X_train = train[['sepal_length','sepal_width','petal_length','petal_width']]
    y_train = train.species
    X_test = test[['sepal_length','sepal_width','petal_length','petal_width']]
    y_test = test.species
    
    mod_dt = DecisionTreeClassifier(max_depth = 3, random_state = 1)
    mod_dt.fit(X_train,y_train)
    prediction=mod_dt.predict(X_test)
    print('The accuracy of the Decision Tree is',"{:.3f}".format(metrics.accuracy_score(prediction,y_test)))
    
    joblib.dump(mod_dt, f"artifacts/{model_name}.joblib")
    
    artifact_dir = BASE_MODEL_ARTIFACT_DIR + "_" + datetime.now().strftime("%Y%m%d_%H%M%S")
    return artifact_dir

In [66]:
from google.cloud import storage
import io

def test_pipeline(model_dir, m_name, data_path):
    model_path = f"{model_dir}/{m_name}.joblib"

    # Download and load model directly from GCS
    client = storage.Client()
    bucket = client.bucket(BUCKET_NAME)
    blob = bucket.blob(model_path)
    buffer = io.BytesIO()
    blob.download_to_file(buffer)
    buffer.seek(0)
    model = joblib.load(buffer)

    data = pd.read_csv(data_path)

    train, test = train_test_split(data, test_size = 0.4, stratify = data['species'], random_state = 42)
    X_train = train[['sepal_length','sepal_width','petal_length','petal_width']]
    y_train = train.species
    X_test = test[['sepal_length','sepal_width','petal_length','petal_width']]

    # Run inference
    preds = model.predict(X_test)
    return preds

In [13]:
# Running the Pipeline for V1 Data
model_name = "model_v1"
data_path = "./data/raw/iris.csv"

In [17]:
MODEL_ARTIFACT_DIR = pipeline(data_path, model_name)
print(MODEL_ARTIFACT_DIR)

The accuracy of the Decision Tree is 0.983
iris_classifier/model_20251102_103422


In [69]:
!gsutil cp artifacts/model_v1.joblib {BUCKET_URI}/{MODEL_ARTIFACT_DIR}/

Copying file://artifacts/model_v1.joblib [Content-Type=application/octet-stream]...
/ [1 files][  2.3 KiB/  2.3 KiB]                                                
Operation completed over 1 objects/2.3 KiB.                                      


In [70]:
print(test_pipeline(MODEL_ARTIFACT_DIR, model_name, data_path))

['virginica' 'setosa' 'versicolor' 'virginica' 'versicolor' 'setosa'
 'virginica' 'versicolor' 'versicolor' 'setosa' 'virginica' 'virginica'
 'setosa' 'virginica' 'versicolor' 'setosa' 'setosa' 'virginica'
 'virginica' 'setosa' 'virginica' 'setosa' 'versicolor' 'virginica'
 'setosa' 'setosa' 'versicolor' 'versicolor' 'virginica' 'setosa' 'setosa'
 'virginica' 'versicolor' 'setosa' 'virginica' 'setosa' 'virginica'
 'virginica' 'setosa' 'virginica' 'versicolor']


In [71]:
# Running the Pipeline for V2 Data
model_name = "model_v2"
data_path = "data/data_v2.csv"

In [72]:
MODEL_ARTIFACT_DIR = pipeline(data_path, model_name)
print(MODEL_ARTIFACT_DIR)

The accuracy of the Decision Tree is 1.000
iris_classifier/model_20251004_173001


In [73]:
!gsutil cp artifacts/model_v2.joblib {BUCKET_URI}/{MODEL_ARTIFACT_DIR}/

Copying file://artifacts/model_v2.joblib [Content-Type=application/octet-stream]...
/ [1 files][  2.2 KiB/  2.2 KiB]                                                
Operation completed over 1 objects/2.2 KiB.                                      


In [74]:
print(test_pipeline(MODEL_ARTIFACT_DIR, model_name, data_path))

['setosa' 'setosa' 'setosa' 'virginica' 'virginica' 'virginica'
 'versicolor' 'virginica' 'setosa' 'versicolor' 'versicolor' 'versicolor'
 'versicolor' 'virginica' 'setosa' 'versicolor' 'versicolor' 'setosa'
 'versicolor' 'virginica']
